# Source- and Recipient-Weighted Evidence Mechanism Probe

- **Project:** Mass-communication opinion-leader model
- **Submodel ID and version:** `SM-OL-WEIGHT-01` 
- **Framework link:** Reviewed `opleader` rough design
- **Probe type:** Mechanism probe
- **Date:** 2026-09-04
- **Status:** Runnable

## 1. Question, Decision, and Framework Link

- **Primary question:** Can recipient-role weighting change leader and ordinary-audience responses beyond differences already produced by their initialized Beta beliefs?
- **Decision supported:** Retain the source-only mechanism as a null and assess a complete source-recipient relation matrix as a competing mechanism before adding production, delivery, and network reach.
- **Shared interfaces:** `Message`, `Exposure`, `MessageAggregation`, `MessageEvidence`, `OpinionUpdate`, and `BetaBelief`.
- **Highest intended claim level:** V1 isolated-mechanism behavior under fixed synthetic inputs.


## 2. Provenance and Boundary Contract

- Katz's research motivates source-mediated personal influence; it does not supply the numerical weights used here.
- The source-recipient matrix is a jointly developed, provisional representation. Its illustrative weights are assumed rather than calibrated.
- Role-specific Beta initializations are experimental boundary conditions, not empirical claims about leaders or ordinary audiences.
- Press production, leader production, delivery, network topology, topic knowledge, and all feedbacks are omitted.
- Each case performs one aggregation and one synchronous opinion update. There is no stochasticity or seed.
- Source and relation weights are dimensionless multipliers of the common base evidence weight.


## 3. Expected Outcomes Before Running

- Under the source-only null, leader and ordinary recipients with matched priors and exposures update identically.
- A relation matrix with identical recipient columns reproduces the source-only null.
- With matched priors, unequal recipient columns can produce different leader and ordinary-audience updates.
- With identical evidence rules, a more concentrated leader prior moves less in its mean than an ordinary prior.
- Evidence weights also increase posterior concentration; passing these checks does not validate any role weight or initialization.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'opinion_model').is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from opinion_model.baseline import (
    belief_from_mean_concentration,
    propose_opinion_update,
)
from opinion_model.core import AgentState, AggregationContext, Exposure, Message
from opinion_model.opleader import (
    OriginatorKind,
    RecipientKind,
    SourceRecipientWeightedAggregation,
    SourceWeightedAggregation,
)

RUN = {
    'submodel_id': 'SM-OL-WEIGHT-01',
    'submodel_version': '0.2',
    'boundary_scenario': 'fixed one-round exposures',
    'base_evidence_weight': 1.0,
    'source_only_weights': {'press': 1.0, 'leader': 3.0},
    'illustrative_relation_weights': {
        'press->leader': 2.0,
        'press->ordinary': 1.0,
        'leader->leader': 1.0,
        'leader->ordinary': 3.0,
    },
    'active_feedbacks': [],
    'omitted_feedbacks': [
        'production', 'delivery', 'network', 'multi-round relay',
        'endogenous role change',
    ],
}
RUN


## 4. Minimal Specification and Implementation

The notebook imports both tested mechanisms from `opinion_model.opleader`. `SourceWeightedAggregation` is the source-only null; `SourceRecipientWeightedAggregation` requires all four source-recipient relations. The role-specific priors below keep initial mean fixed while varying concentration, so initialization effects are distinguishable from relation-weight effects. Higher Beta concentration is interpreted here as greater prior confidence, not automatically as stubbornness or public commitment.


In [ ]:
PRESS_ID = 10
LEADER_SOURCE_ID = 1
RECIPIENT_ID_BY_KIND = {
    RecipientKind.LEADER: 0,
    RecipientKind.ORDINARY: 2,
}
ORIGINATOR_KIND_BY_ID = {
    PRESS_ID: OriginatorKind.PRESS,
    LEADER_SOURCE_ID: OriginatorKind.LEADER,
}
RECIPIENT_KIND_BY_ID = {
    recipient_id: recipient_kind
    for recipient_kind, recipient_id in RECIPIENT_ID_BY_KIND.items()
}
SOURCE_ONLY_WEIGHTS = {
    OriginatorKind.PRESS: RUN['source_only_weights']['press'],
    OriginatorKind.LEADER: RUN['source_only_weights']['leader'],
}
RELATION_WEIGHTS = {
    (OriginatorKind.PRESS, RecipientKind.LEADER):
        RUN['illustrative_relation_weights']['press->leader'],
    (OriginatorKind.PRESS, RecipientKind.ORDINARY):
        RUN['illustrative_relation_weights']['press->ordinary'],
    (OriginatorKind.LEADER, RecipientKind.LEADER):
        RUN['illustrative_relation_weights']['leader->leader'],
    (OriginatorKind.LEADER, RecipientKind.ORDINARY):
        RUN['illustrative_relation_weights']['leader->ordinary'],
}

SOURCE_ONLY = SourceWeightedAggregation(
    originator_kind_by_id=ORIGINATOR_KIND_BY_ID,
    source_weight_by_kind=SOURCE_ONLY_WEIGHTS,
)
RELATION_AWARE = SourceRecipientWeightedAggregation(
    originator_kind_by_id=ORIGINATOR_KIND_BY_ID,
    recipient_kind_by_id=RECIPIENT_KIND_BY_ID,
    relation_weight_by_kinds=RELATION_WEIGHTS,
)
MECHANISMS = {
    'source-only': SOURCE_ONLY,
    'relation-aware': RELATION_AWARE,
}
INITIALIZATIONS = {
    'matched uncertainty': {
        RecipientKind.LEADER: {'mean': 0.5, 'concentration': 4.0},
        RecipientKind.ORDINARY: {'mean': 0.5, 'concentration': 4.0},
    },
    'leader more concentrated': {
        RecipientKind.LEADER: {'mean': 0.5, 'concentration': 40.0},
        RecipientKind.ORDINARY: {'mean': 0.5, 'concentration': 4.0},
    },
}

def make_exposure(consumer_id, producer_id, stance, label):
    return Exposure(
        round_index=1,
        consumer_id=consumer_id,
        message=Message(
            message_id=f'r1:{label}',
            round_index=1,
            producer_id=producer_id,
            stance=stance,
        ),
    )

def evaluate_case(initialization, recipient_kind, mechanism, messages):
    consumer_id = RECIPIENT_ID_BY_KIND[recipient_kind]
    exposures = tuple(
        make_exposure(consumer_id, producer_id, stance, label)
        for producer_id, stance, label in messages
    )
    evidence = MECHANISMS[mechanism](
        exposures, AggregationContext(RUN['base_evidence_weight'])
    )
    prior_spec = INITIALIZATIONS[initialization][recipient_kind]
    before = AgentState(
        belief_from_mean_concentration(**prior_spec)
    )
    after = propose_opinion_update(before, evidence)
    return {
        'initialization': initialization,
        'recipient': recipient_kind.value,
        'mechanism': mechanism,
        'weighted_support': evidence.weighted_support,
        'weighted_oppose': evidence.weighted_oppose,
        'mean_before': before.belief.mean,
        'mean_after': after.belief.mean,
        'mean_change': after.belief.mean - before.belief.mean,
        'absolute_mean_change': abs(after.belief.mean - before.belief.mean),
        'concentration_before': before.belief.concentration,
        'concentration_after': after.belief.concentration,
    }


## 5. Deterministic and Boundary Checks

These cases verify no-input invariance, the nesting of the source-only null inside the relation-aware representation, all four source-recipient relations, and the separation of relation weights from Beta-prior concentration.


In [ ]:
CONFLICT_MESSAGES = [
    (PRESS_ID, 1, 'press-support'),
    (LEADER_SOURCE_ID, -1, 'leader-oppose'),
]
empty = evaluate_case(
    'matched uncertainty', RecipientKind.LEADER, 'relation-aware', []
)
assert empty['mean_after'] == empty['mean_before']
assert empty['concentration_after'] == empty['concentration_before']

source_equivalent_relations = {
    (originator_kind, recipient_kind): SOURCE_ONLY_WEIGHTS[originator_kind]
    for originator_kind in OriginatorKind
    for recipient_kind in RecipientKind
}
source_equivalent = SourceRecipientWeightedAggregation(
    originator_kind_by_id=ORIGINATOR_KIND_BY_ID,
    recipient_kind_by_id=RECIPIENT_KIND_BY_ID,
    relation_weight_by_kinds=source_equivalent_relations,
)
for recipient_kind in RecipientKind:
    consumer_id = RECIPIENT_ID_BY_KIND[recipient_kind]
    exposures = tuple(
        make_exposure(consumer_id, producer_id, stance, label)
        for producer_id, stance, label in CONFLICT_MESSAGES
    )
    context = AggregationContext(RUN['base_evidence_weight'])
    assert source_equivalent(exposures, context) == SOURCE_ONLY(exposures, context)

relation_rows = []
for producer_id, originator_kind in ORIGINATOR_KIND_BY_ID.items():
    for recipient_kind in RecipientKind:
        result = evaluate_case(
            'matched uncertainty',
            recipient_kind,
            'relation-aware',
            [(producer_id, 1, f'{originator_kind.value}-{recipient_kind.value}')],
        )
        expected = RELATION_WEIGHTS[(originator_kind, recipient_kind)]
        assert result['weighted_support'] == expected
        relation_rows.append({
            'source': originator_kind.value,
            'recipient': recipient_kind.value,
            'effective_weight': result['weighted_support'],
        })

null_leader = evaluate_case(
    'matched uncertainty', RecipientKind.LEADER, 'source-only', CONFLICT_MESSAGES
)
null_ordinary = evaluate_case(
    'matched uncertainty', RecipientKind.ORDINARY, 'source-only', CONFLICT_MESSAGES
)
relation_leader = evaluate_case(
    'matched uncertainty', RecipientKind.LEADER, 'relation-aware', CONFLICT_MESSAGES
)
relation_ordinary = evaluate_case(
    'matched uncertainty', RecipientKind.ORDINARY, 'relation-aware', CONFLICT_MESSAGES
)
concentrated_leader = evaluate_case(
    'leader more concentrated',
    RecipientKind.LEADER,
    'source-only',
    CONFLICT_MESSAGES,
)
assert null_leader['mean_after'] == null_ordinary['mean_after']
assert relation_leader['mean_after'] > 0.5 > relation_ordinary['mean_after']
assert concentrated_leader['absolute_mean_change'] < null_leader['absolute_mean_change']

relation_checks = pd.DataFrame(relation_rows)
relation_checks


## 6. Exploratory Experiment

The factorial comparison crosses mechanism (`source-only` or `relation-aware`), recipient role, and initialization. Both roles receive the same fixed reinforcing or conflicting messages. The matched initialization isolates relation weights; the leader-concentrated initialization isolates one possible prior-confidence difference. No parameter is calibrated.


In [ ]:
scenarios = {
    'reinforcement: press + / leader +': [
        (PRESS_ID, 1, 'press-support'),
        (LEADER_SOURCE_ID, 1, 'leader-support'),
    ],
    'conflict: press + / leader -': [
        (PRESS_ID, 1, 'press-support'),
        (LEADER_SOURCE_ID, -1, 'leader-oppose'),
    ],
}

rows = []
for initialization in INITIALIZATIONS:
    for recipient_kind in RecipientKind:
        for mechanism in MECHANISMS:
            for scenario_name, messages in scenarios.items():
                row = evaluate_case(
                    initialization, recipient_kind, mechanism, messages
                )
                rows.append({'scenario': scenario_name, **row})

results = pd.DataFrame(rows)
conflict_results = results.query("scenario == 'conflict: press + / leader -'")
matched_null = conflict_results.query(
    "initialization == 'matched uncertainty' and mechanism == 'source-only'"
)
assert matched_null['mean_after'].nunique() == 1

matched_relation = conflict_results.query(
    "initialization == 'matched uncertainty' and mechanism == 'relation-aware'"
).set_index('recipient')
assert matched_relation.loc['leader', 'mean_after'] > 0.5
assert matched_relation.loc['ordinary', 'mean_after'] < 0.5

concentrated_null = conflict_results.query(
    "initialization == 'leader more concentrated' and mechanism == 'source-only'"
).set_index('recipient')
assert (
    concentrated_null.loc['leader', 'absolute_mean_change']
    < concentrated_null.loc['ordinary', 'absolute_mean_change']
)
results.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
recipient_order = ['leader', 'ordinary']
for ax, initialization in zip(axes, INITIALIZATIONS):
    group = conflict_results.query('initialization == @initialization')
    for mechanism, mechanism_group in group.groupby('mechanism'):
        ordered = mechanism_group.set_index('recipient').loc[recipient_order]
        ax.plot(
            recipient_order,
            ordered['mean_after'],
            marker='o',
            label=mechanism,
        )
    ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--')
    ax.set(title=initialization, xlabel='Recipient role')
axes[0].set_ylabel('Posterior mean after press + / leader -')
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()


## 7. Results and Conditional Interpretation

The next cell reports the observed deterministic result. Interpretation remains conditional on the fixed-message boundary and does not establish empirical validity.


In [ ]:
matched_null_by_role = matched_null.set_index('recipient')
matched_relation_by_role = matched_relation
concentrated_null_by_role = concentrated_null
display(Markdown(
    f"""
- **Source-only null:** With matched priors, the leader and ordinary recipient both end at `{matched_null_by_role.loc['leader', 'mean_after']:.3f}` under the same conflicting exposures. Recipient role has no effect.
- **Initialization effect:** Keeping the source-only null but increasing only leader prior concentration changes the leader result to `{concentrated_null_by_role.loc['leader', 'mean_after']:.3f}`, while the ordinary result remains `{concentrated_null_by_role.loc['ordinary', 'mean_after']:.3f}`.
- **Relation effect:** With matched priors but the illustrative relation matrix, the leader ends at `{matched_relation_by_role.loc['leader', 'mean_after']:.3f}` and the ordinary recipient at `{matched_relation_by_role.loc['ordinary', 'mean_after']:.3f}`.
- **Highest completed level:** V1 behavior of source-only and source-recipient aggregation coupled to the existing Beta update under fixed inputs.
- **Supports:** The implementation separates recipient-role weighting from role-specific prior concentration and makes all four communication relations explicit.
- **Does not support:** Any empirical relation weight, role-specific prior, network effect, population outcome, or claim that leaders are inherently more confident or resistant.
"""
))


## 8. Disposition and Change Impact

- **Disposition:** Retain `SourceWeightedAggregation` as the explicit source-only benchmark; retain `SourceRecipientWeightedAggregation` as a provisional competing mechanism.
- **Initialization status:** Matched priors are the null. Higher leader concentration is an assumed scenario for probing prior confidence, not a confirmed leader property. Initial mean and concentration should be varied separately.
- **Affected mechanism:** `opleader` message aggregation and this V1 notebook.
- **Unaffected:** Baseline aggregation and the shared `MessageAggregation` protocol; production, selection, delivery, network, and platform mechanisms remain omitted.
- **Open semantic decision:** Whether leader stability represents knowledge/confidence, behavioral rigidity, public commitment, or a combination. Beta concentration directly represents only the first interpretation.
- **Required later check:** V2 coupling only after role-specific initialization, press and leader production, and delivery interfaces are specified.
